## Дерево решений для качества вина

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

data = pd.read_csv('winequality-red.csv', sep=';').dropna()
data['quality_class'] = pd.cut(data['quality'], bins=[0, 5, 6, 10], labels=['низкое', 'среднее', 'высокое'])
X = data.drop(columns=['quality', 'quality_class'])
y = data['quality_class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

models = {
    'По умолчанию': DecisionTreeClassifier(random_state=42),
    'Ограниченная глубина': DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=42),
    'Энтропия': DecisionTreeClassifier(criterion='entropy', max_depth=5, min_samples_leaf=5, random_state=42)
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    results.append([name, accuracy_score(y_test, prediction)])
print(pd.DataFrame(results, columns=['Модель', 'Accuracy']))

best_model = max(models.values(), key=lambda model: accuracy_score(y_test, model.predict(X_test)))
prediction = best_model.predict(X_test)
print(classification_report(y_test, prediction))

plt.figure(figsize=(18, 8))
plot_tree(best_model, feature_names=X.columns, class_names=[str(x) for x in best_model.classes_], filled=True, max_depth=3, fontsize=7)
plt.show()

importance = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Важность признаков:')
print(importance)
importance.plot(kind='bar', figsize=(10, 4))
plt.show()

                 Модель  Accuracy
0          По умолчанию  0.684375
1  Ограниченная глубина  0.640625
2              Энтропия  0.603125
              precision    recall  f1-score   support

     высокое       0.56      0.58      0.57        43
      низкое       0.76      0.76      0.76       149
     среднее       0.64      0.63      0.64       128

    accuracy                           0.68       320
   macro avg       0.65      0.66      0.65       320
weighted avg       0.68      0.68      0.68       320

Важность признаков:
alcohol                 0.186828
sulphates               0.118393
volatile acidity        0.104154
total sulfur dioxide    0.091889
pH                      0.087723
free sulfur dioxide     0.078495
fixed acidity           0.078311
residual sugar          0.067273
chlorides               0.067158
density                 0.064448
citric acid             0.055327
dtype: float64


/tmp/ipykernel_5581/2154716470.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_5581/2154716470.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Ответы на контрольные вопросы

1. Дерево решений — модель, которая последовательно делит данные по условиям над признаками и выдаёт прогноз в листовых узлах.
2. При обучении выбираются такие признак и порог, которые сильнее всего уменьшают неоднородность дочерних узлов. Деление повторяется до достижения ограничений остановки.
3. Для классификации часто используют Gini и entropy, для регрессии — squared error, Friedman MSE, absolute error и Poisson deviance.
4. Основные гиперпараметры: `criterion`, `splitter`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`, `max_leaf_nodes`, `class_weight` и `ccp_alpha`.